# Lecture Notes: Statistics and Machine Learning for Data Analysis

MFRE Coding Workshop, follow-up notes for Day 2 and the Python applications session.

These notes exist because the code in Day 2 and the applications notebook moved
faster than the ideas behind it. Here we slow down. Each idea is laid out the same
way:

1. A definition, in one or two sentences.
2. A small worked example you can check by hand.
3. The Python code that does it.
4. A plain explanation of what the numbers mean.

You do not need to memorize anything. Read top to bottom, run each cell, and stop
to predict the output before you run it. That habit is what turns code you can copy
into code you understand.

Run the setup cell below first. Everything after it depends on it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

# Two datasets you already met.
# gapminder: one row per country per year, with income and life expectancy.
gap = pd.read_csv('../data/gapminder_gni.csv').dropna(subset=['gdpPercap', 'lifeExp'])

# policy: the climate-pledge dataset from the applications session.
policy = pd.read_csv('../stat applications/r_applications/policy_analysis.csv')

print('gapminder:', gap.shape)
print('policy   :', policy.shape)

---
# Part 1: Describing data

Before you model anything, you describe it. A single column of numbers is hard to
read. Descriptive statistics compress that column into a few numbers that tell you
where the data sits and how spread out it is.

We will use the 13 provincial temperatures from the Day 2 weather data as the
running example, because 13 numbers are few enough to check by hand.

In [ ]:
temps = np.array([-6, -1, -18, -20, 0, 0, -12, -4, -2, -17, -13, -16, -16], dtype=float)
print('the 13 temperatures:', temps)
print('how many          :', len(temps))

## 1.1 Mean, median, mode: where is the center?

**Definition.**
- The **mean** is the arithmetic average: add every value, divide by the count.
- The **median** is the middle value once the data is sorted. Half the values sit
  below it, half above.
- The **mode** is the value that appears most often.

**Example.** For the five numbers 8, 9, 10, 11, 12 the mean is 10 and the median is
also 10. Now add a sixth value of 200. The mean jumps to 41.7, but the median barely
moves, to 10.5. One extreme value drags the mean a long way and leaves the median
almost untouched.

In [ ]:
small = np.array([8, 9, 10, 11, 12], dtype=float)
print('mean  :', small.mean(), '  median:', np.median(small))

small_outlier = np.append(small, 200)
print('with 200 added ->  mean:', round(small_outlier.mean(), 1),
      '  median:', np.median(small_outlier))

In [ ]:
# The three centers of the real temperature data:
print('mean  :', round(temps.mean(), 4))     # -9.6154
print('median:', np.median(temps))           # -12.0
# Mode: the value that occurs most. pandas makes this easy.
print('mode  :', pd.Series(temps).mode().tolist())   # 0 and -16 both appear twice

**Plain explanation.** The mean is what most people call the average, and it is
the right choice when the data is roughly symmetric. The median is the better choice
when the data is skewed or has outliers, because it does not get dragged around by a
few extreme values. The mode is mostly useful for categories, for example the most
common income group, rather than for continuous measurements.

The gap between the mean and median is itself information. Here the mean is -9.6 and
the median is -12.0. The mean sits above the median because a few provinces near 0
pull the average up. Treat a large gap as a prompt to go and look at the shape of the
data, not as proof of skew on its own. These 13 temperatures actually have a skew of
about +0.09, which is close to symmetric. The gap here comes from the values clustering
in two loose groups rather than from a long tail. Section 1.6 gives you the statistic
that settles the question.

## 1.2 Variance and standard deviation: how spread out is it?

Two datasets can share the same mean and look nothing alike. `[10, 10, 10]` and
`[0, 10, 20]` both average to 10, but the second is far more spread out. Variance and
standard deviation measure that spread.

**Definition.** The **variance** is the sum of squared distances from the mean,
divided by `n - 1`:

$$ \text{var} = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})^2 $$

The **standard deviation** is the square root of the variance. We take the square
root so the answer comes back in the original units. If the data is in degrees, the
standard deviation is in degrees, while the variance is in degrees squared.

We divide by `n - 1` rather than `n` for a sample. That is Bessel's correction, and
it stops the sample from underestimating the true spread. It is why pandas `.var()`
and `.std()` divide by `n - 1` by default.

**Example.** For `[0, 10, 20]` the mean is 10. The squared distances are 100, 0, 100.
Their sum is 200, and dividing by `n - 1 = 2` gives a variance of 100 and a standard
deviation of 10.

In [ ]:
demo = np.array([0, 10, 20], dtype=float)
mean = demo.mean()
squared_distances = (demo - mean) ** 2
print('squared distances:', squared_distances)          # [100.   0. 100.]
print('variance         :', squared_distances.sum() / (len(demo) - 1))   # 100.0
print('std deviation    :', np.sqrt(100))               # 10.0

# pandas does all of that in one call:
print('pandas var       :', pd.Series(demo).var())      # 100.0
print('pandas std       :', pd.Series(demo).std())      # 10.0

In [ ]:
# The spread of the real temperatures:
s = pd.Series(temps)
print('mean :', round(s.mean(), 3))    # -9.615
print('var  :', round(s.var(), 3))     # 57.756
print('std  :', round(s.std(), 3))     # 7.600

**Plain explanation.** A standard deviation of 7.6 degrees means a typical province
sits about 7.6 degrees away from the average of -9.6. Small standard deviation means
the values huddle near the mean. Large standard deviation means they are scattered.

This is the number that was missing when the applications notebook printed a variance
of 76 million for the non-high-income group. That did not mean the group averaged 76
million. It meant the values were wildly spread out, because one absurd outlier was
still in the data: Kiribati, recorded at -86,515. For comparison the high-income group,
whose worst value is Latvia at -472, has a variance of 5,267, four orders of magnitude
smaller. A giant variance is a red flag to go looking for outliers, which is the next
topic.

## 1.3 Range, quartiles, and the IQR

**Definition.**
- The **range** is simply max minus min.
- **Quartiles** cut the sorted data into four equal parts. The first quartile (Q1) is
  the value below which 25 percent of the data falls, the second (Q2) is the median,
  and the third (Q3) is the 75 percent mark.
- The **interquartile range (IQR)** is Q3 minus Q1. It is the width of the middle half
  of the data, and unlike the range it ignores the extremes.

**Example.** For the temperatures, Q1 is -16 and Q3 is -2, so the middle half of the
provinces sit between -16 and -2 degrees, an IQR of 14 degrees.

In [ ]:
s = pd.Series(temps)
print('min, max :', s.min(), s.max(), '  range:', s.max() - s.min())
q1, q2, q3 = s.quantile([0.25, 0.50, 0.75])
print('Q1, Q2, Q3:', q1, q2, q3)          # -16.0  -12.0  -2.0
print('IQR       :', q3 - q1)             # 14.0

# describe() reports all of this at once:
print(s.describe())

**Plain explanation.** `describe()` is the first thing to run on any numeric column.
It hands you the count, mean, standard deviation, the min and max, and the three
quartiles in one call. The IQR matters because it is a measure of spread that outliers
cannot inflate. It is also the machinery behind the boxplot in Part 2.

## 1.4 Outliers: one value can break your summary

**Definition.** An **outlier** is a value far away from the rest of the data. It may be
a genuine extreme case, or it may be a data-entry error. Either way it deserves
attention, because summary statistics like the mean and variance are very sensitive to
it.

**Example.** In the applications session, the high-income countries' pledge column had
Latvia recorded at -472, far below any other high-income country. The non-high-income
group had a worse one still, Kiribati at -86,515. Each dominated its own group's mean.
A common rule of thumb flags any value more than 1.5 IQRs below Q1 or above Q3 as a
potential outlier.

In [ ]:
# A clean group with one planted outlier, the same shape as the Latvia problem.
group = np.array([3, 5, -2, 4, 1, 6, 0, 2], dtype=float)
print('mean without outlier:', group.mean())            # 2.375

group_bad = np.append(group, -472)
print('mean WITH -472      :', round(group_bad.mean(), 3))   # -50.333
print('median WITH -472    :', np.median(group_bad))         # 2.0, barely moves

In [ ]:
# The 1.5-IQR rule, applied to the contaminated group:
s = pd.Series(group_bad)
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
low_fence = q1 - 1.5 * iqr
high_fence = q3 + 1.5 * iqr
print('fences:', round(low_fence, 2), 'to', round(high_fence, 2))
print('flagged as outliers:', s[(s < low_fence) | (s > high_fence)].tolist())   # [-472.0]

**Plain explanation.** One value out of nine moved the mean from 2.4 to -50, while the
median barely moved, from 2.5 to 2.0. That is the whole lesson: the mean is not robust, the
median is. When a mean looks impossible, sort the data or draw a boxplot and look for
the culprit. Then decide, on purpose, whether to keep it, drop it, or report both. Never
delete a value just because it is inconvenient, but never let one bad row speak for the
whole dataset either.

## 1.5 Population, sample, and the standard error

**Definition.**
- The **population** is every unit you care about, for example every country on earth.
- A **sample** is the subset you actually measured.
- A **parameter** is a number describing the population. A **statistic** is the same number
  computed from your sample. You almost never know the parameter, so you estimate it with
  the statistic.
- The **standard error (SE)** of the mean is how much the sample mean would bounce around
  if you drew a new sample. It is the standard deviation divided by the square root of n:

$$ \text{SE} = \frac{s}{\sqrt{n}} $$

**Example.** Life expectancy across 1320 country-years has a standard deviation of 12.28
years, but the standard error of the mean is only 0.34 years. Those two numbers answer
different questions.

In [ ]:
life = gap['lifeExp']
n = len(life)

print('n                     :', n)                       # 1320
print('standard deviation    :', round(life.std(), 3))    # 12.282  spread of the DATA
print('standard error of mean:', round(life.std() / np.sqrt(n), 4))   # 0.3380  precision of the MEAN

**Plain explanation.** This is the single most confused pair in statistics, so be precise.

- The **standard deviation** describes how spread out the individual values are. It does not
  shrink when you collect more data, because the world is as varied as it is.
- The **standard error** describes how precisely you know the *mean*. It shrinks as n grows,
  because averaging more observations pins the average down more tightly.

Note the square root: to halve the standard error you need four times as much data. That is
why large samples give precise averages but never make the underlying variation disappear.

The standard error is the engine inside every hypothesis test in Part 4. The "std err" column
of the regression summary is exactly this idea applied to a coefficient.

## 1.6 The shape of a distribution: normal, skew, kurtosis

**Definition.**
- The **normal distribution**, or bell curve, is symmetric around its mean, with most values
  near the middle and thin tails. Many statistical methods assume approximate normality.
- **Skewness** measures asymmetry. Zero is symmetric, positive means a long right tail,
  negative means a long left tail.
- **Kurtosis** measures tail heaviness. Pandas reports *excess* kurtosis, where 0 matches the
  normal curve. Positive means heavier tails and more extreme values than normal.

**Example.** Life expectancy has a skew of -0.34, so it is close to symmetric with a slight
left tail. Raw income has a skew of 2.86 and excess kurtosis of 15.94, which is strongly
right-skewed with very heavy tails. Take the log of income and the skew drops to 0.04, almost
perfectly symmetric. That is the numerical version of what the histogram showed you.

In [ ]:
print('lifeExp   skew:', round(gap['lifeExp'].skew(), 3),
      ' excess kurtosis:', round(gap['lifeExp'].kurt(), 3))     # -0.341, -1.071
print('gdpPercap skew:', round(gap['gdpPercap'].skew(), 3),
      ' excess kurtosis:', round(gap['gdpPercap'].kurt(), 3))   #  2.861, 15.940
print('log10(gdp) skew:', round(np.log10(gap['gdpPercap']).skew(), 3))   # 0.036, nearly symmetric

**Plain explanation.** Skew tells you which way the tail points, kurtosis tells you how
heavy the tails are. Both matter because a lot of standard machinery assumes roughly normal
data. When income came back with skew 2.86, that was the signal to transform it. Taking the
logarithm pulled the tail in and brought the skew to almost zero, which is why the log version
fits a straight line so much better.

This is also how to read the Skew and Kurtosis lines in the regression diagnostics. They are
describing the *residuals*, not the raw data, and they are asking whether the leftover errors
look like a normal bell curve.

## 1.7 Standardizing: z-scores and percentiles

**Definition.** A **z-score** rewrites a value as how many standard deviations it sits from
the mean:

$$ z = \frac{x - \bar{x}}{s} $$

A z of 0 is exactly average, +2 is two standard deviations above, -1.5 is one and a half
below. Standardizing puts different variables on a common scale so they can be compared.

A **percentile** answers a different question: what share of the data falls below this value.
The 90th percentile is the value that 90 percent of observations sit under.

**Example.** After standardizing life expectancy, the values run from -3.07 to +1.73, the mean
is 0 and the standard deviation is 1 by construction. About 98.6 percent of values fall within
2 standard deviations of the mean. The 90th percentile of life expectancy is 75.76 years.

In [ ]:
life = gap['lifeExp']
z = (life - life.mean()) / life.std()

print('z range :', round(z.min(), 2), 'to', round(z.max(), 2))    # -3.07 to 1.73
print('z mean  :', round(z.mean(), 6), ' z std:', round(z.std(), 3))   # ~0 and 1.0
print('within 2 sd:', round((z.abs() <= 2).mean(), 3))            # 0.986
print('90th percentile of lifeExp:', round(life.quantile(0.90), 2))    # 75.76

**Plain explanation.** Standardizing is how you compare apples to oranges. A country cannot
be "3 units better" at both income and life expectancy, because the units differ. Convert both
to z-scores and the comparison becomes meaningful.

Two practical uses. First, z-scores are a common outlier rule: anything beyond roughly plus or
minus 3 is unusual. Second, many machine learning methods require features on a similar scale,
and standardizing is the usual fix. That step is called **feature scaling**, and you will meet
it again in Section 3.12.

## 1.8 Covariance: the raw version of correlation

**Definition.** **Covariance** measures whether two variables move together, by averaging the
product of their distances from their own means. Positive covariance means they rise together,
negative means one rises as the other falls.

The problem with covariance is that its size depends on the units. Change income from dollars
to thousands of dollars and the covariance changes, even though the relationship has not.
**Correlation is covariance divided by both standard deviations**, which cancels the units and
forces the answer into the range -1 to 1.

**Example.** The covariance between log income and life expectancy is 5.51, a number that is
hard to interpret on its own. The correlation is 0.81, which immediately reads as a strong
positive relationship.

In [ ]:
log_gdp = np.log10(gap['gdpPercap'])
life = gap['lifeExp']

covariance = np.cov(log_gdp, life)[0, 1]
correlation = log_gdp.corr(life)

print('covariance :', round(covariance, 4))     # 5.5061, units are hard to read
print('correlation:', round(correlation, 4))    # 0.8149, unit free

# correlation is just the scaled covariance:
print('cov / (sd * sd):', round(covariance / (log_gdp.std() * life.std()), 4))   # 0.8149

**Plain explanation.** Covariance and correlation carry the same information about direction.
Correlation is the version you can actually interpret, because it is scale free. This is why you
almost always report correlation. Covariance still matters under the hood: it is what regression
and most multivariate methods are built from.

---
# Part 2: Seeing data

A table of numbers hides its own shape. A picture shows it in a second. Three plots
cover most of what you need at the start of any analysis.

## 2.1 Histogram: the shape of one variable

**Definition.** A **histogram** slices the range of one numeric variable into bins and
draws a bar for how many values fall in each bin. It shows you the distribution: where
the data piles up, whether it is symmetric or skewed, and whether there are gaps.

**Example.** Income per person across all countries and years is heavily skewed. Most
of the world sits at low income, and a long thin tail stretches out to a few very rich
country-years. A histogram makes that shape obvious immediately.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(gap['gdpPercap'], bins=40, color='steelblue', edgecolor='white')
plt.xlabel('GDP per capita')
plt.ylabel('Number of country-years')
plt.title('Histogram of GDP per capita (right-skewed)')
plt.show()

**Plain explanation.** The tall bars on the left and the long tail to the right are
the signature of right-skew. This is exactly the case where the mean and median
disagree, because the tail pulls the mean to the right. Whenever you are about to report
a mean, draw the histogram first. If it looks like this, report the median too, or take
a logarithm to pull the tail in.

## 2.2 Boxplot: spread and outliers at a glance

**Definition.** A **boxplot** draws the five-number summary as a picture. The box spans
Q1 to Q3 (the IQR), the line inside the box is the median, the whiskers reach out to the
last values inside 1.5 IQRs, and anything past the whiskers is drawn as an individual
point, a flagged outlier.

**Example.** Split life expectancy by continent and the boxplot shows, in one frame,
which continents are higher, which are more spread out, and which have outlying
countries.

In [ ]:
continents = ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania']
data_by_cont = [gap.loc[gap['continent'] == c, 'lifeExp'] for c in continents]

plt.figure(figsize=(8, 4))
plt.boxplot(data_by_cont, tick_labels=continents)
plt.ylabel('Life expectancy (years)')
plt.title('Life expectancy by continent')
plt.show()

**Plain explanation.** Read a boxplot from the box outward. A tall box means the
middle half of the data is widely spread. A median line sitting low inside its box means
the data is skewed. The dots beyond the whiskers are the outliers the 1.5-IQR rule
flags, drawn for you automatically. This is the fastest way to compare several groups at
once and to spot the extreme values before they wreck a mean.

## 2.3 Scatter plot: the relationship between two variables

**Definition.** A **scatter plot** puts one variable on the x axis and another on the y
axis and draws one dot per observation. It is how you see whether two variables move
together, and in which direction.

**Example.** Plot life expectancy against income per person. There is a clear upward
pattern: richer country-years tend to live longer. The pattern is curved, which is why
later we will put income on a log scale to straighten it out.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(gap['gdpPercap'], gap['lifeExp'], s=10, alpha=0.4, color='darkgreen')
plt.xlabel('GDP per capita')
plt.ylabel('Life expectancy (years)')
plt.title('Life expectancy vs income')
plt.show()

**Plain explanation.** A scatter plot answers three questions at once. Is there a
relationship? Here, yes. Which direction? Upward, so higher income goes with higher life
expectancy. What shape? Curved, rising steeply at first and then flattening. The curve
is the reason a straight-line model will fit better after we take a logarithm of income.
Always plot two variables before you try to model their relationship. The plot tells you
what kind of model is even reasonable.

**Which plot to reach for.** Histogram for the distribution of one variable. Boxplot to
compare that distribution across groups, or to hunt for outliers. Scatter plot for the
relationship between two numeric variables. Those three cover the large majority of
first looks at a dataset.

---
# Part 3: Relationships and the machine learning idea

We now move from describing data to modeling it. A model is a rule learned from data
that lets you say something about values you have not seen. That single sentence is the
whole idea behind machine learning, and regression is the simplest example of it.

## 3.1 Correlation: how strongly do two variables move together?

**Definition.** The **correlation coefficient**, written r, is a single number between
-1 and 1 that measures the strength and direction of a straight-line relationship.

- r near 1: strong positive, the two rise together.
- r near -1: strong negative, one rises as the other falls.
- r near 0: no straight-line relationship.

**Example.** Income and life expectancy have a correlation of about 0.63 on the raw
scale. After taking the base-10 logarithm of income, the correlation rises to about
0.81, because the log transform straightens the curve you saw in the scatter plot, and
correlation only measures the straight-line part.

In [ ]:
gap['loggdp'] = np.log10(gap['gdpPercap'])

print('corr(income, lifeExp)      :', round(gap['gdpPercap'].corr(gap['lifeExp']), 4))   # 0.6271
print('corr(log income, lifeExp)  :', round(gap['loggdp'].corr(gap['lifeExp']), 4))      # 0.8149

**Plain explanation.** Correlation is a summary of a scatter plot in one number, but
it only sees straight lines. A strong curved relationship can have a middling
correlation, which is exactly why the number jumped from 0.63 to 0.81 once we straightened
the curve with a log.

The warning that goes with correlation: **correlation is not causation.** Two variables
can move together because one causes the other, because a third variable drives both, or
by pure coincidence in a small sample. Correlation measures association. It says nothing
about what causes what. Ice-cream sales and drowning deaths correlate, because both rise
in summer, not because ice cream is dangerous.

## 3.2 What a model is, and the machine learning idea

**Definition.** A **model** is a rule that takes inputs and returns a prediction. In
supervised machine learning you hand the computer example inputs paired with known
answers, and it adjusts the rule until its predictions match the answers as closely as
possible. That adjustment is called **fitting** or **training**.

The inputs have several names you will see used interchangeably: features, predictors,
independent variables, or X. The answer you are predicting is the target, the outcome,
the dependent variable, or y.

The point of a model is not to memorize the data you already have. It is to generalize,
to make good predictions on new data. That is why we later hold some data back and test
on it.

## 3.3 Linear regression: the line of best fit

**Definition.** **Linear regression** fits a straight line through the data:

$$ y = b_0 + b_1 x $$

`b_0` is the intercept, the predicted y when x is zero. `b_1` is the slope, how much y
changes when x goes up by one unit. Fitting the model means choosing `b_0` and `b_1`.

**Example.** Predict life expectancy from log income. The fitted line turns out to be
`lifeExp = -3.99 + 18.19 * log10(income)`. The slope of 18.19 says that every tenfold
increase in income (one step of 1 on the log-10 scale) is associated with about 18 more
years of life expectancy.

In [ ]:
X = sm.add_constant(gap[['loggdp']])   # add_constant creates the intercept column
y = gap['lifeExp']

model = sm.OLS(y, X).fit()
print('intercept b0:', round(model.params['const'], 3))    # -3.985
print('slope     b1:', round(model.params['loggdp'], 3))   # 18.191

In [ ]:
# Draw the fitted line on top of the data.
plt.figure(figsize=(7, 5))
plt.scatter(gap['loggdp'], y, s=10, alpha=0.3, color='darkgreen', label='data')

xs = np.linspace(gap['loggdp'].min(), gap['loggdp'].max(), 100)
ys = model.params['const'] + model.params['loggdp'] * xs
plt.plot(xs, ys, color='red', linewidth=2, label='fitted line')

plt.xlabel('log10(GDP per capita)')
plt.ylabel('Life expectancy (years)')
plt.title('Linear regression of life expectancy on log income')
plt.legend()
plt.show()

**Plain explanation.** The red line is the model. Once you have the intercept and slope,
you can predict life expectancy for any income by plugging its log into the equation. The
line does not touch every point, and it is not supposed to. It is the single straight line
that comes closest to all the points at once. The next section says what closest means.

## 3.4 Least squares: what best fit means

**Definition.** For each data point, the **residual** is the vertical gap between the
actual y and the line's prediction: `residual = actual - predicted`. Ordinary least
squares chooses the intercept and slope that make the **sum of the squared residuals**
as small as possible.

We square the residuals for two reasons. Squaring makes every gap positive, so gaps above
and below the line do not cancel out. And squaring punishes large misses far more than
small ones, so the line pays most attention to the points it is furthest from.

**Example.** Take Canada in 2007. Its actual life expectancy was 80.7 years and the line
predicts 79.0, so the residual is 80.7 - 79.0 = +1.7. The model was slightly pessimistic
for Canada. Least squares picked the one line, out of every possible line, whose residuals
squared and added together are the smallest they can be.

In [ ]:
predictions = model.predict(X)
residuals = y - predictions

print('first five actual   :', y.head().round(1).tolist())
print('first five predicted:', predictions.head().round(1).tolist())
print('first five residuals:', residuals.head().round(1).tolist())
print('sum of squared residuals:', round((residuals ** 2).sum(), 1))

# The Canada 2007 row from the example above:
gap_r = gap.reset_index(drop=True)
gap_r['predicted'] = model.predict(X).values
gap_r['residual'] = gap_r['lifeExp'] - gap_r['predicted']
canada = gap_r[gap_r['country'] == 'Canada'].iloc[-1]
print()
print('Canada', int(canada['year']),
      ': actual', round(canada['lifeExp'], 1),
      '| predicted', round(canada['predicted'], 1),
      '| residual', round(canada['residual'], 1))

# The largest miss in the whole dataset, to show residuals are not all small:
worst = gap_r.loc[gap_r['residual'].idxmin()]
print('worst miss:', worst['country'], int(worst['year']),
      '| actual', round(worst['lifeExp'], 1),
      '| predicted', round(worst['predicted'], 1),
      '| residual', round(worst['residual'], 1))

**Plain explanation.** Fitting is an optimization: search over all possible lines and
keep the one with the smallest total squared error. You do not do that search by hand.
`sm.OLS(...).fit()` solves it exactly in one step. What you need to carry away is that the
line is a compromise that balances all the residuals, and that squaring is why a few large
misses can pull the line noticeably.

## 3.5 Error metrics: RMSE and MAE

**Definition.** Two numbers summarize how wrong a model's predictions are, on average.

- **MAE**, mean absolute error, is the average size of the residuals ignoring sign:
  $$ \text{MAE} = \frac{1}{n}\sum |y_i - \hat{y}_i| $$
- **RMSE**, root mean squared error, squares the residuals, averages them, then takes the
  square root:
  $$ \text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2} $$

Both are in the same units as y, so both are directly readable. RMSE is always at least as
large as MAE, and it is larger when a few predictions are badly off, because squaring
weights big misses more heavily.

**Example.** For the life expectancy model, MAE is about 5.3 years and RMSE is about 7.1
years. So the model is off by roughly 5 years on a typical country, but the larger RMSE
warns that some countries are missed by much more than that.

In [ ]:
mae = residuals.abs().mean()
rmse = np.sqrt((residuals ** 2).mean())

print('MAE :', round(mae, 4), 'years')    # 5.3379
print('RMSE:', round(rmse, 4), 'years')   # 7.1166

**Plain explanation.** MAE answers "how far off am I on average". RMSE answers the same
question but leans on the big mistakes. If being badly wrong occasionally is much worse than
being slightly wrong often, watch RMSE. If every error counts the same, MAE is the more
honest summary. Report them in the units of the target so you can judge whether the error is
tolerable: an RMSE of 7 years means something you can reason about, where a bare R-squared
does not.

## 3.6 Training and predicting: why you hold data back

**Definition.** A model can look excellent on the data it was fitted to and still be
useless, because it has partly memorized that specific data. To get an honest estimate of
how it will do on new data, you split the rows into a **training set** and a **test set**.
You fit the model on the training set only, then measure its error on the test set, which
it has never seen.

**Example.** Split the gapminder rows 70 percent for training and 30 percent for testing.
The model's RMSE is about 7.0 on the training data and about 7.3 on the held-out test data.
The two numbers being close is good news: it means the model generalizes and did not just
memorize the training rows.

In [ ]:
rng = np.random.default_rng(0)          # fixed seed so the split is reproducible
order = rng.permutation(len(gap))
cut = int(0.70 * len(gap))
train_idx, test_idx = order[:cut], order[cut:]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

fitted = sm.OLS(y_train, X_train).fit()          # learn from training rows only

train_rmse = np.sqrt(((y_train - fitted.predict(X_train)) ** 2).mean())
test_rmse = np.sqrt(((y_test - fitted.predict(X_test)) ** 2).mean())

print('training rows:', len(train_idx), ' test rows:', len(test_idx))
print('train RMSE:', round(train_rmse, 3))   # 7.023
print('test  RMSE:', round(test_rmse, 3))    # 7.331

**Plain explanation.** The test error is the number that matters, because it estimates
performance on data the model will actually face. If the training error were tiny and the
test error large, the model would be **overfitting**: memorizing noise in the training data
instead of learning the real pattern. Here the two are close, so the straight-line model is
capturing a genuine relationship rather than memorizing. This train-then-test discipline is
the backbone of applied machine learning, from this simple line up to the largest models.

## 3.7 Reading a regression summary

`statsmodels` prints a dense table. You do not need every entry. Six of them carry most of
the meaning.

- **coef**: the intercept and slope, the b values.
- **std err**: how uncertain each coefficient is. Smaller is more precise.
- **t**: the coefficient divided by its standard error. Larger in size means the
  coefficient is more clearly different from zero.
- **P>|t|**: the p-value for that coefficient, explained fully in Part 4. Below about 0.05
  is the usual bar for calling a coefficient statistically significant.
- **R-squared**: the share of the variation in y that the model explains, between 0 and 1.
- **Adj. R-squared**: R-squared with a penalty for each predictor added, so it can fall,
  and even go negative, when a predictor pulls its weight less than random noise would.

In [ ]:
print(model.summary())

**Plain explanation.** Read the table in this order. First look at R-squared to see how
much the model explains: here it is about 0.66, so log income accounts for roughly two
thirds of the variation in life expectancy. Then look at the slope row: the coefficient is
18.19, its p-value is 0.000, so the relationship is very unlikely to be an accident of the
sample. The intercept of -3.99 is not meaningful on its own here, because an income whose
log is zero (one dollar per person) is outside anything the data contains. A coefficient is
only worth interpreting over the range of x you actually observed.

## 3.8 The full OLS summary, field by field

The summary table above has three blocks: a header of overall model numbers, the
coefficient table in the middle, and a block of diagnostics at the bottom. Most people
learn the coefficient table and ignore the rest. Here is what every field means, using
the life-expectancy model you just fitted. Run the cell so the table is in front of you.

In [ ]:
print(model.summary())

### Block 1: the header (overall model)

- **Dep. Variable**: the target you are predicting. Here `lifeExp`.
- **Model** and **Method**: `OLS` and `Least Squares`, the technique from Section 3.4.
- **No. Observations**: the number of rows used, 1320 here. Rows with missing values were
  dropped, so check this matches what you expect.
- **Df Model**: degrees of freedom for the model, which is the number of predictors not
  counting the intercept. It is 1 here, because log income is the only predictor.
- **Df Residuals**: observations minus estimated coefficients, 1320 - 2 = 1318. This is the
  budget of independent information left over after fitting, and it feeds the p-values.
- **Covariance Type**: how the standard errors were computed. `nonrobust` is the default and
  assumes constant error spread. `HC3`, used in the applications notebook, relaxes that
  assumption and is safer when the spread of the residuals changes with x.

- **R-squared**: the share of the variation in y the model explains, 0 to 1. Here 0.664, so
  log income explains about 66 percent of the variation in life expectancy.
- **Adj. R-squared**: R-squared penalized for the number of predictors, 0.664 here. It barely
  differs from R-squared because there is only one predictor. It matters when you have many.
- **F-statistic**: a single test of whether the model as a whole beats predicting the mean.
  Here 2604.6, very large.
- **Prob (F-statistic)**: the p-value for that F-test. Here it prints as essentially zero, so
  the model as a whole is highly significant. This is the first number to check: if it is not
  small, no individual coefficient is worth reading.
- **Log-Likelihood**, **AIC**, **BIC**: scores for comparing competing models on the same data.
  Log-likelihood is higher for a better fit. AIC and BIC are lower for a better fit and add a
  penalty for extra predictors. You do not interpret any of them alone. You compare them across
  models, and the model with the lower AIC or BIC is preferred.

### Block 2: the coefficient table (one row per term)

Each predictor, plus the constant, gets a row with six numbers.

- **coef**: the estimated intercept and slopes. Here the constant is -3.985 and the slope on
  log income is 18.191. The slope says a tenfold rise in income (one unit of log-10) goes with
  about 18 more years of life expectancy.
- **std err**: the standard error of that coefficient, how much it would wobble across repeated
  samples. Smaller means more precise. The slope's is 0.356, tight relative to the coefficient.
- **t**: the coefficient divided by its standard error, 18.191 / 0.356 = 51.0. It measures how
  many standard errors the estimate sits away from zero. Large in size means clearly nonzero.
- **P>|t|**: the p-value for the null that this coefficient is truly zero. The slope's is 0.000,
  so the relationship is very unlikely to be an accident of the sample. The constant's is 0.002.
- **[0.025 and 0.975]**: the 95 percent confidence interval for the coefficient. For the slope it
  is 17.49 to 18.89. Read it as: values in this range are consistent with the data. If that
  interval does not contain zero, the coefficient is significant at the 5 percent level, which is
  the same message the p-value gives, shown as a range instead of a single number.

### Block 3: the diagnostics (are the assumptions holding?)

OLS trusts a few assumptions. This block checks them. You do not act on every number, but you
should know what each one is warning about.

- **Omnibus** and **Prob(Omnibus)**: a test of whether the residuals are normally distributed.
  A small Prob(Omnibus), here 0.000, says they are not perfectly normal. With 1320 rows this is
  common and rarely fatal, but do not ignore it.
- **Skew**: the asymmetry of the residuals. Zero is symmetric. Here -0.73, a mild left lean.
- **Kurtosis**: the heaviness of the tails. A normal distribution scores 3. Here 4.08, so the
  residuals have slightly heavier tails than normal, meaning a few larger misses than a perfect
  bell curve would produce.
- **Durbin-Watson**: checks whether consecutive residuals are correlated. It runs from 0 to 4,
  and about 2 means no correlation. Here 0.44, far from 2, which flags correlation between nearby
  rows. That is expected: the data has repeated countries over years, so rows are not independent.
- **Jarque-Bera (JB)** and **Prob(JB)**: a second normality test built from the skew and kurtosis.
  It agrees with the Omnibus here, tiny p-value, residuals not normal.
- **Cond. No.**: the condition number, a check for multicollinearity, meaning predictors that are
  near-duplicates of each other. Small is good. Here 25.8, low, so no problem. In the applications
  notebook it was 1.6 billion, because GDP and GDP squared were almost the same column, and that is
  what made the squared term's standard error explode.

**Plain explanation.** Read the summary top to bottom in three passes. First the header: is
Prob(F-statistic) small, and how much does R-squared explain? Second the coefficient table: what is
each slope, and is its p-value below 0.05? Third the diagnostics, as a sanity check: a wild condition
number means multicollinearity, a Durbin-Watson far from 2 means correlated rows, strong non-normal
residuals mean the p-values are approximate. None of the diagnostics here sink the model, but they do
tell you to lean on the robust `HC3` standard errors and to remember that repeated countries make the
rows less independent than OLS assumes.

## 3.9 The baseline: is your model beating "do nothing"?

**Definition.** A **baseline** is the dumbest reasonable prediction, used as a yardstick. For a
numeric target the usual baseline is: ignore all predictors and predict the mean for every row.
A model that cannot beat its baseline is not adding anything.

**Example.** Predicting the mean life expectancy for every country-year gives an RMSE of 12.28
years. The regression gives 7.12. That is a 42 percent reduction in error, so the model is
genuinely earning its place.

In [ ]:
baseline_pred = y.mean()                               # predict the same number every time
baseline_rmse = np.sqrt(((y - baseline_pred) ** 2).mean())
model_rmse = np.sqrt(((y - model.predict(X)) ** 2).mean())

print('baseline RMSE (predict the mean):', round(baseline_rmse, 3))   # 12.277
print('model RMSE                      :', round(model_rmse, 3))      # 7.117
print('error reduction                 :', str(round(100 * (1 - model_rmse / baseline_rmse), 1)) + '%')

**Plain explanation.** Always compute the baseline before you celebrate a model. An RMSE of 7
years sounds bad in isolation and good next to a baseline of 12. R-squared is doing the same job
in a different form: an R-squared of 0.664 says the model removed about two thirds of the squared
error the baseline left behind. This is also why a negative adjusted R-squared is so damning. It
means the model lost to predicting the mean.

## 3.10 Underfitting, overfitting, and the bias-variance tradeoff

**Definition.**
- **Underfitting**: the model is too simple to capture the real pattern. It does poorly on the
  training data and equally poorly on new data.
- **Overfitting**: the model is too flexible and memorizes the noise in the training data. It
  does very well on training data and badly on new data.
- **Bias** is error from wrong assumptions, the underfitting direction. **Variance** is error
  from being too sensitive to the particular training sample, the overfitting direction. The
  **bias-variance tradeoff** is that reducing one tends to raise the other, and the best model
  sits in between.

**Example.** Take 30 points from a straight-line relationship with noise added, so we know the
true answer. Shuffle them, fit polynomials of increasing flexibility to 20 of them, then score
on the 10 held out. Training error falls as the degree rises, every time. Test error does not.

In [ ]:
# A straight-line relationship with noise added, so we know the truth.
rng = np.random.default_rng(42)
x_all = np.linspace(0, 10, 30)
y_all = 2.5 * x_all + 5 + rng.normal(0, 4, size=len(x_all))

# Shuffle before splitting. Splitting a sorted x by position would put the whole test
# set beyond the training range, and we would be measuring extrapolation, not overfitting.
shuffled = np.random.default_rng(7).permutation(len(x_all))
train_i, test_i = shuffled[:20], shuffled[20:]
x_tr, y_tr = x_all[train_i], y_all[train_i]      # 20 training points
x_te, y_te = x_all[test_i], y_all[test_i]        # 10 held-out points

for degree in (1, 2, 9):
    coeffs = np.polyfit(x_tr, y_tr, degree)
    train_rmse = np.sqrt(np.mean((y_tr - np.polyval(coeffs, x_tr)) ** 2))
    test_rmse = np.sqrt(np.mean((y_te - np.polyval(coeffs, x_te)) ** 2))
    print(f'degree {degree}: train RMSE {train_rmse:8.2f} | test RMSE {test_rmse:12.2f}')

# degree 1: train 3.14 | test 2.70   <- about right, matches the true straight line
# degree 2: train 3.12 | test 2.79
# degree 9: train 2.41 | test 8.27   <- best on training data, worst on new data

**Plain explanation.** Look at the two columns separately. Training RMSE falls steadily from
3.14 to 2.41 as the model gets more flexible, exactly as you would expect: more freedom means it
can hug the training points more tightly. Test RMSE moves the other way, from 2.70 up to 8.27.
The degree-9 curve bends hard to chase the scatter in the training points, and those bends are
fitted to noise, so they do not carry over to the held-out points.

The lesson is blunt: **training error always improves with complexity, so it cannot tell you when
to stop.** Only held-out data can. If someone shows you a model evaluated on the data it was fitted
to, you have learned nothing about whether it works.

## 3.11 Cross-validation: a more reliable estimate

**Definition.** A single train/test split depends on luck: an unusual test set gives a misleading
score. **K-fold cross-validation** fixes that. Split the data into k equal folds. Train on k-1 of
them and test on the one left out. Repeat so each fold serves as the test set once, then average
the k scores.

**Example.** Five-fold cross-validation on the life expectancy model gives RMSEs of 6.90, 7.07,
6.95, 7.21 and 7.48. The average is 7.12 with a spread of 0.21, so the estimate is stable across
splits and we can trust it more than any single number.

In [ ]:
indices = np.arange(len(gap))
rng = np.random.default_rng(0)
rng.shuffle(indices)
folds = np.array_split(indices, 5)          # five roughly equal folds

scores = []
for k in range(5):
    test_i = folds[k]
    train_i = np.concatenate([folds[j] for j in range(5) if j != k])
    fold_model = sm.OLS(y.iloc[train_i], X.iloc[train_i]).fit()
    rmse_k = np.sqrt(((y.iloc[test_i] - fold_model.predict(X.iloc[test_i])) ** 2).mean())
    scores.append(float(rmse_k))          # float() keeps the printout clean

print('fold RMSEs :', [round(s, 3) for s in scores])
print('mean CV RMSE:', round(np.mean(scores), 3))     # 7.122
print('spread (sd) :', round(np.std(scores), 3))      # 0.210

**Plain explanation.** Cross-validation buys you two things. A more reliable average score,
because every row gets used for testing exactly once, and a sense of how much that score wobbles.
A small spread, as here, means the model behaves consistently. A large spread is a warning that
your result depends heavily on which rows happened to land in the test set, which usually means
you do not have enough data.

Five or ten folds are the common choices. The cost is that you fit the model k times instead of
once, which matters only when fitting is slow.

## 3.12 Machine learning vocabulary

You will meet these words constantly. Here they are in one place, with the meaning that matters.

**The setup**

| Term | Meaning |
|---|---|
| **Supervised learning** | You have labelled examples: inputs paired with known answers. Regression here is supervised. |
| **Unsupervised learning** | No answers given. You look for structure, for example clustering countries into groups. |
| **Regression task** | The target is a number, such as life expectancy in years. |
| **Classification task** | The target is a category, such as high income yes or no. |
| **Features (X)** | The inputs. Also called predictors, independent variables, covariates. |
| **Target (y)** | What you are predicting. Also called the label, outcome, dependent variable. |
| **Observation** | One row. Also called an instance, a sample, or a data point. |

**Fitting and evaluating**

| Term | Meaning |
|---|---|
| **Training** | Adjusting the model so its predictions match the known answers. Also called fitting. |
| **Parameters** | Numbers the model learns from data, for example the slope and intercept. |
| **Hyperparameters** | Settings you choose before training, for example the polynomial degree or the number of folds. The model does not learn these. |
| **Generalization** | How well the model performs on data it has never seen. The only thing that really matters. |
| **Training set / test set** | Rows used to fit, and rows held back to score honestly. |
| **Validation set** | A third split used to tune hyperparameters, so the test set stays untouched until the very end. |
| **Baseline** | The simplest sensible prediction, used as the yardstick. |

**Things that go wrong**

| Term | Meaning |
|---|---|
| **Underfitting** | Too simple. Bad on training and test data alike. |
| **Overfitting** | Too flexible. Great on training data, bad on new data. |
| **Bias-variance tradeoff** | Simple models have high bias, flexible models have high variance. The best model balances the two. |
| **Data leakage** | Information from the test set sneaks into training, for example scaling using the full dataset before splitting. Scores look great and are fake. |
| **Multicollinearity** | Two predictors carry nearly the same information, for example GDP and GDP squared. Coefficients become unstable and standard errors inflate. |
| **Class imbalance** | In classification, one category dominates, so predicting the majority every time looks deceptively accurate. |

**Common preparation steps**

| Term | Meaning |
|---|---|
| **Feature scaling** | Putting features on a comparable scale, usually with z-scores. Needed by methods based on distance. |
| **Feature engineering** | Building new predictors from existing ones, for example taking the log of income or squaring GDP. |
| **Dummy variables** | Turning a category into 0/1 columns so a model can use it. Also called one-hot encoding. |
| **Imputation** | Filling missing values rather than dropping the rows. |

**Plain explanation.** Two of these deserve emphasis because they are where real analyses fail
quietly. **Data leakage** produces beautiful results that collapse in practice: if you standardize
using the mean of the entire dataset and only then split, your training set has already seen the
test set. Compute anything of that kind on the training data alone. And **multicollinearity** is
what wrecked the Kuznets regression in the applications notebook. GDP and GDP squared were nearly
the same column, the condition number hit 1.6 billion, and the squared term's standard error
inflated so much that a real effect could not have been detected either way.

---
# Part 4: Hypothesis testing

This is the part of the applications session that lost people. The code was three lines,
but the reason for running it was never spelled out. Here is the reason.

## 4.1 The core idea: signal or noise?

You measure a difference. High-income countries pledge more than others, say, or Canada
emits more than the world average. The question is always the same: **is this difference
real, or could it easily have come from random chance?**

Any sample wobbles. Draw 20 people and their average height will not equal the true
average exactly, just from luck of the draw. So when you see a difference between two
groups, some of it is real signal and some is random noise. A hypothesis test is a formal
way to ask how much of what you see could be noise alone.

**Why we need it.** Without a test you are guessing. A difference might look large but sit
well within what chance produces, or look small but be far beyond chance. Eyeballing cannot
tell those apart. The test puts a number on it.

## 4.2 Null and alternative hypotheses

**Definition.** Every test starts with two competing statements.

- The **null hypothesis** (H0) is the boring explanation: there is no real effect, and any
  difference you see is just noise.
- The **alternative hypothesis** (H1) is the claim you are actually interested in: there is
  a real effect.

You assume the null is true, then ask how surprising your data would be under that
assumption. If the data would be very surprising when the null holds, you reject the null in
favor of the alternative. You never prove the null. You either reject it or fail to reject
it, the way a court finds guilty or not guilty, never innocent.

**Example.** For "are countries signing all three climate pledges?", the null is that the
average country signed all 3, so the mean equals 3. The alternative is that the average is
less than 3.

## 4.3 The p-value: what it means and what it does not

**Definition.** The **p-value** is the probability of seeing data at least as extreme as
yours **if the null hypothesis were true**. A small p-value means your data would be very
unlikely under the null, which is evidence against the null.

The common threshold is 0.05. If the p-value is below 0.05 you reject the null and call the
result statistically significant. The 0.05 is a convention, not a law of nature.

**What the p-value is NOT.** This is where people go wrong, so read it twice.

- It is **not** the probability that the null is true.
- It is **not** the probability that your result happened by chance.
- A small p-value does **not** mean the effect is large or important. With enough data, a
  trivial difference can be highly significant.
- A large p-value does **not** prove the null is true. It only means this sample cannot
  distinguish the effect from zero.

## 4.4 Significance level, one-tailed and two-tailed

**Definition.** The **significance level**, written alpha, is the threshold you fix in
advance, usually 0.05. It is the risk you are willing to accept of rejecting a null that is
actually true (a false positive).

A **two-tailed** test asks whether the value differs from the null in either direction,
higher or lower. A **one-tailed** test asks about one specific direction only. Use one-tailed
only when you genuinely care about a single direction and decided so before seeing the data.
The pledge test is one-tailed because the only interesting question is whether countries sign
*fewer* than 3, not more, since 3 is the maximum.

## 4.5 Confidence intervals: the estimate plus its uncertainty

**Definition.** A **confidence interval** is a range of plausible values for the quantity you
are estimating. A 95 percent interval is built by a procedure that captures the true value 95
percent of the time across repeated samples.

The usual form is the estimate plus or minus roughly two standard errors:

$$ \bar{x} \pm t \times \text{SE} $$

**Example.** Mean life expectancy in the sample is 61.31 years with a standard error of 0.34.
The 95 percent confidence interval runs from 60.64 to 61.97 years.

In [ ]:
life = gap['lifeExp']
n = len(life)
mean = life.mean()
se = life.std() / np.sqrt(n)

low, high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
print('sample mean :', round(mean, 3))            # 61.305
print('standard err:', round(se, 4))              # 0.3380
print('95% interval: [', round(low, 3), ',', round(high, 3), ']')   # [60.642, 61.969]

**Plain explanation.** A confidence interval says more than a p-value, because it shows both
the size of the effect and how precisely you know it. A narrow interval means a precise estimate.
A wide one means you should not lean on the number.

The link to hypothesis testing is direct: **if a 95 percent interval excludes the null value, the
two-sided test rejects at the 5 percent level.** They are the same statement in different clothes.
That is why the regression summary prints `[0.025  0.975]` next to each coefficient. An interval
for a slope that does not contain zero corresponds to a p-value below 0.05.

One caution on wording. The interval is a statement about the *procedure*, not about this one
interval. Saying "there is a 95 percent chance the true mean is in here" is the common informal
phrasing, and it is not strictly what the method guarantees.

## 4.6 Type I and Type II errors, and power

**Definition.** A test can be wrong in two directions.

| | Null is actually true | Null is actually false |
|---|---|---|
| **You reject the null** | Type I error (false positive) | Correct |
| **You fail to reject** | Correct | Type II error (false negative) |

- A **Type I error** means you announce an effect that is not there. Its probability is alpha,
  the significance level you chose, usually 0.05.
- A **Type II error** means you miss a real effect. Its probability is called beta.
- **Power** is 1 minus beta: the chance of detecting an effect that genuinely exists. Power rises
  with a larger sample, a larger true effect, and less noise.

**Example.** Setting alpha to 0.05 means accepting a 5 percent false-positive rate when the null
is true. Lowering it to 0.01 buys fewer false positives but raises the chance of missing a real
effect. You cannot reduce both at once with a fixed sample. Getting more data is what improves
both together.

**Plain explanation.** This table explains why the proportion test in Section 4.8 failing to
find a difference is not proof of no difference. With only 55 high-income countries and both
signing rates above 96 percent, that test has very little power. A real but modest difference could
easily hide there, and a Type II error would look exactly like the result we saw.

The practical habit: when a test comes back not significant, ask whether the study could have
detected the effect at all. Report the confidence interval alongside the p-value. If the interval
is wide and includes both zero and effects large enough to matter, the honest conclusion is "this
data cannot tell", not "there is no effect".

## 4.7 The t-test: comparing means

The t-test is the workhorse. It comes in two forms.

### One-sample t-test
**Definition.** Tests whether the mean of one group differs from a fixed reference number.

**Example.** Did countries sign all 3 pledges on average? Reference value 3, one-tailed
(less than). The result: mean 1.663 pledges across 184 countries, t = -21.6, p = 1.7e-52.
The p-value is far below 0.05, so we reject the null decisively. Countries signed 1.66 of 3
on average, not 3.

In [ ]:
h1 = policy[['country', 'partytopledge1', 'partytopledge2', 'partytopledge3']].dropna()
h1['total'] = h1[['partytopledge1', 'partytopledge2', 'partytopledge3']].sum(axis=1)
h1 = h1.groupby('country').first()

t_stat, p_value = stats.ttest_1samp(h1['total'], popmean=3, alternative='less')
print('n countries :', len(h1))                  # 184
print('mean pledges:', round(h1['total'].mean(), 3))   # 1.663
print('t statistic :', round(t_stat, 3))         # -21.591
print('p value     :', p_value)                  # 1.67e-52 -> reject the null

### Two-sample t-test
**Definition.** Tests whether two independent groups have different means. The Welch version,
`equal_var=False`, does not assume the two groups have equal variance, and is the safe default.

**Example.** Do high-income and other countries differ in emissions per person? High-income
mean is 14.06 tonnes across 69 countries, the rest average 4.11 across 129, t = 6.51,
p = 6.0e-09. Far below 0.05, so the difference is real: high-income countries emit
substantially more per person.

In [ ]:
d = policy[['country', 'year', 'Incomegroup', 'GHG_percapita']].dropna()
d = d.loc[d.groupby('country')['year'].idxmax()]      # most recent year per country

high = d.loc[d['Incomegroup'] == 'High income', 'GHG_percapita']
rest = d.loc[d['Incomegroup'] != 'High income', 'GHG_percapita']

t_stat, p_value = stats.ttest_ind(high, rest, equal_var=False)   # Welch
print('high income : n =', len(high), ' mean =', round(high.mean(), 3))   # 69, 14.064
print('the rest    : n =', len(rest), ' mean =', round(rest.mean(), 3))   # 129, 4.111
print('t statistic :', round(t_stat, 3))    # 6.513
print('p value     :', p_value)             # 5.98e-09 -> reject the null

**Plain explanation of the t-statistic.** The t-statistic is the size of the difference
measured in units of its own uncertainty. A t near 0 means the difference is small compared
to the noise. A t far from 0, positive or negative, means the difference is large compared to
the noise. The p-value turns that t into a probability. The sign of t only tells you the
direction: negative in the pledge test because the mean was below 3, positive in the emissions
test because high-income came out above the rest.

## 4.8 Proportion z-test: comparing shares

**Definition.** When the outcome is yes or no rather than a number, you compare **proportions**
instead of means. The two-proportion z-test asks whether the share of yes differs between two
groups.

**Example.** Are high-income countries more likely to sign the Paris Agreement? High-income
signed at a rate of 0.9636 (55 countries), the rest at 0.9603 (126). z = 0.107, p = 0.915.

The p-value is nowhere near 0.05, so we **fail to reject the null**. There is no evidence of a
difference. The two rates differ by 0.003, which is a third of one percentage point.
In counts that is 53 of 55 high-income countries against 121 of 126 others.

In [ ]:
h3 = policy[['country', 'Incomegroup', 'partytopledge3']].dropna().groupby('country').first()
hi = h3.loc[h3['Incomegroup'] == 'High income', 'partytopledge3']
lo = h3.loc[h3['Incomegroup'] != 'High income', 'partytopledge3']

z_stat, p_value = sm.stats.proportions_ztest([hi.sum(), lo.sum()], [len(hi), len(lo)])
print('high income share:', round(hi.mean(), 4), ' n =', len(hi))   # 0.9636, 55
print('other share      :', round(lo.mean(), 4), ' n =', len(lo))   # 0.9603, 126
print('z statistic:', round(z_stat, 3))     # 0.107
print('p value    :', round(p_value, 3))    # 0.915 -> fail to reject

**Plain explanation.** Failing to reject is a real, useful result, not a failure of the
analysis. It says this data cannot tell the two groups apart. Note the reason it is hard here:
both shares are above 96 percent, so almost everyone signed and there is barely any variation
left to explain. When a variable is nearly constant, no test will find group differences in it,
regardless of sample size.

## 4.9 ANOVA: comparing more than two groups

**Definition.** A t-test compares two groups. **ANOVA** (analysis of variance) compares three
or more at once. Its null hypothesis is that every group has the same mean. It reports an
**F-statistic** and a p-value.

**Example.** Do average emissions per person differ across the seven World Bank regions?
F = 4.55, p = 0.00024. Below 0.05, so we reject the null: at least one region differs from at
least one other.

In [ ]:
h5 = policy[['country', 'year', 'wbregion', 'GHG_percapita']].dropna()
h5 = h5.loc[h5.groupby('country')['year'].idxmax()]

groups = [g['GHG_percapita'].values for _, g in h5.groupby('wbregion')]
f_stat, p_value = stats.f_oneway(*groups)
print('number of regions:', len(groups))    # 7
print('F statistic:', round(f_stat, 3))      # 4.547
print('p value    :', p_value)               # 0.000239 -> reject the null

**Plain explanation.** ANOVA answers only "are they all the same?" When it says no, it does
not tell you *which* regions differ. That needs a follow-up (a post-hoc test such as Tukey's).
Two cautions apply here: the groups are very unbalanced, from 4 countries in North America to 50
in Europe, and ANOVA assumes roughly equal spread across groups, which per-capita emissions
almost certainly violate. The small p-value survives some of that, but the assumptions are worth
stating out loud rather than hiding.

## 4.10 The traps

A short list of mistakes that catch people, drawn straight from the applications notebook.

**Significance is not importance.** A p-value tells you an effect is probably real. It says
nothing about whether it is big enough to care about. With a large sample, a difference too
small to matter can be highly significant. Always look at the size of the effect, the actual
means and their difference, alongside the p-value.

**Adjusted R-squared can go negative.** Plain R-squared never falls when you add a predictor,
even a useless one, so it rewards throwing in variables. Adjusted R-squared penalizes each
predictor. When it goes negative, as it did for the province regression in the Day 2 notes, the
model is doing worse than predicting the mean for everyone. That is the model telling you it has
failed, and reading it correctly is a real skill.

**Failing to reject is not proving the null.** A large p-value means this sample cannot detect an
effect. The effect may still exist and simply be too small, or the sample too small, to see. Absence
of evidence is not evidence of absence.

**One test per question.** Running many tests and reporting only the ones that came out below 0.05
is called p-hacking. If you run 20 tests on random data, one will clear 0.05 on average by chance
alone. Decide your question before you look, and count every test you ran.

---
# Summary

- Describe before you model: the mean says where, the standard deviation says how spread out,
  and the median and IQR are the versions that outliers cannot break.
- Draw the data: histogram for one variable, boxplot to compare groups and find outliers, scatter
  for a relationship.
- A model is a rule learned from data to predict new data. Linear regression fits the line that
  minimizes the squared residuals. RMSE and MAE report the error in real units. Fit on training
  data and measure on held-out test data so the error estimate is honest.
- A hypothesis test asks whether a difference is signal or noise. The p-value is the chance of data
  this extreme if there were no real effect. Small p-value, reject the null. It measures evidence,
  not size, and never proves the null.

Run every cell, change the numbers, and see what moves. The fastest way to trust a method is to
watch it react when you feed it something you understand.